# 🕹️ Deep RL Evolution Pipeline — REINFORCE → A2C → PPO

**Environment:** `ALE/Pong-v5` from raw pixels  
**Runtime:** GPU (T4 recommended) | Runtime → Change runtime type → GPU

---

## 1 · Install packages and verify setup.

In [ ]:
# System packages
!apt-get update -qq
!apt-get install -y -qq xvfb ffmpeg

# Python packages
!pip install -q \
    "setuptools<82" \
    jedi \
    "gymnasium[atari]" \
    "autorom[accept-rom-license]" \
    opencv-python-headless \
    pyvirtualdisplay \
    matplotlib \
    pandas \
    imageio \
    imageio-ffmpeg

# Install Atari ROMs
!AutoROM --accept-license

import sys, gymnasium as gym, ale_py, torch, numpy as np, cv2, matplotlib, imageio, setuptools

print('=' * 65)
print(f'Python {sys.version.split()[0]} | Gymnasium {gym.__version__} | ALE {ale_py.__version__}')
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Verify ALE works
gym.register_envs(ale_py)
env = gym.make('ALE/Pong-v5')
obs, _ = env.reset()
print(f'ALE/Pong-v5 obs shape: {obs.shape} | actions: {env.action_space}')
env.close()
print('=' * 65)
print('✅ Setup complete')

## 2 · Clone repo and set working directory.

In [ ]:
import os, sys

REPO_URL  = 'https://github.com/NatnaelTigistu/rl-atari-evolution.git'
REPO_NAME = 'rl-atari-evolution'

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
else:
    !git -C {REPO_NAME} pull

ROOT = os.path.abspath(REPO_NAME)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)

print(f'📂 Working directory: {os.getcwd()}')

## 3 · Start virtual display.

In [ ]:
from visualization.display import init_virtual_display

_display = init_virtual_display()
print('Virtual display ready ✓')

## 4 · Verify wrapper chain shape and dtype.

In [ ]:
import numpy as np
from config import CFG
from src.wrappers import make_atari_env

env = make_atari_env(CFG.ENV_NAME, seed=CFG.SEED)
obs, info = env.reset(seed=CFG.SEED)
obs_arr = np.asarray(obs)

print(f'Environment : {CFG.ENV_NAME}')
print(f'Obs shape   : {obs_arr.shape}   (expect (4, 84, 84))')
print(f'Obs dtype   : {obs_arr.dtype}  (expect float32)')
print(f'Obs range   : [{obs_arr.min():.3f}, {obs_arr.max():.3f}]  (expect [0.0, 1.0])')
print(f'Action space: {env.action_space}')

obs2, reward, terminated, truncated, _ = env.step(env.action_space.sample())
print(f'Step API    : obs={np.asarray(obs2).shape}, terminated={terminated}, truncated={truncated}  ✓')
env.close()
print('\n✅ Wrapper chain OK')

## 5 · Train REINFORCE.

In [ ]:
!python -m src.train \
    --algo reinforce \
    --episodes 1000 \
    --save-freq 50 \
    --log-interval 10 \
    --seed 42

## 6 · Plot reward and loss curves.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('logs/reinforce_rewards.csv')
WINDOW = 50
df['moving_avg'] = df['reward'].rolling(WINDOW, min_periods=1).mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(df['episode'], df['reward'],     alpha=0.3, color='steelblue', label='Episode reward')
axes[0].plot(df['episode'], df['moving_avg'], color='navy', linewidth=2, label=f'{WINDOW}-ep avg')
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Reward')
axes[0].set_title('REINFORCE — Training Rewards')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(df['episode'], df['loss'], alpha=0.5, color='tomato', label='Policy loss')
axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Loss')
axes[1].set_title('REINFORCE — Policy Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('logs/reinforce_training_curve.png', dpi=120)
plt.show()

## 7 · Load checkpoint and record gameplay video.

In [ ]:
import glob, os

checkpoints = sorted(
    glob.glob('checkpoints/reinforce/ep_*.pt'),
    key=lambda p: int(os.path.basename(p).replace('ep_', '').replace('.pt', ''))
)
if not checkpoints:
    checkpoints = glob.glob('checkpoints/reinforce/final.pt')

if not checkpoints:
    print('⚠️  No checkpoint found — run the training cell first.')
else:
    latest_ckpt = checkpoints[-1]
    print(f'Latest checkpoint: {latest_ckpt}')

In [ ]:
import os, glob
import torch
import numpy as np
import gymnasium as gym
from IPython.display import Video, display as ipy_display

from config import CFG
from src.wrappers import make_eval_env
from src.reinforce import REINFORCEAgent

DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
VIDEO_DIR = 'videos/reinforce'
MAX_STEPS = 20 * 30  # 20 s at ~30 fps

os.makedirs(VIDEO_DIR, exist_ok=True)

eval_env   = make_eval_env(CFG.ENV_NAME, seed=0, render_mode='rgb_array')
action_dim = eval_env.action_space.n

agent = REINFORCEAgent(action_dim=action_dim, device=DEVICE)
agent.load(latest_ckpt)
agent.policy.eval()

rec_env = gym.wrappers.RecordVideo(
    eval_env,
    video_folder=VIDEO_DIR,
    episode_trigger=lambda ep: ep == 0,
    name_prefix='reinforce_eval',
    video_length=MAX_STEPS,
)

obs, _    = rec_env.reset(seed=0)
done      = False
total_rew = 0.0
step      = 0

while not done and step < MAX_STEPS:
    obs_t = torch.tensor(
        np.asarray(obs, dtype=np.float32), dtype=torch.float32
    ).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        action, _ = agent.policy.get_action(obs_t)

    obs, reward, terminated, truncated, _ = rec_env.step(action)
    total_rew += float(reward)
    done = terminated or truncated
    step += 1

rec_env.close()
print(f'Episode done — steps: {step}  total reward: {total_rew:.1f}')

videos = sorted(glob.glob(f'{VIDEO_DIR}/*.mp4'))
if videos:
    ipy_display(Video(videos[-1], embed=True, width=420))
else:
    print('⚠️  No .mp4 found — ensure ffmpeg is installed.')

## 8 · Train A2C or PPO.

In [ ]:
!python -m src.train --algo a2c --episodes 3000 --save-freq 100 --log-interval 10 --seed 42